# Annotate train questions 201–400 and merge with questions 1–200

The existing class annotation covers the first 200 unique training questions from both the 0.5B and 3B candidate banks. This notebook selects the **next 200 unique questions**, annotates their candidates with the same judge rubric, then merges both blocks into one 400-question annotation and pair dataset.

The question offset is applied to unique question IDs, not raw JSONL rows, so each selected question keeps its complete candidate bank.

In [ ]:
from pathlib import Path
import json, os, sys, hashlib

PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
sys.path.insert(0, str(PROJECT_ROOT))
from cse_dpo.candidate_bank_class_judge import (
    CandidateBankClassJudge, JUDGE_SYSTEM, build_pairs, digest, prepare_records
)

BANK_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/candidate_banks/strict_extractive_qwen25_05b_3b_all_splits_s20_v2_seed3407'
BANK_FILES = {
    '05b': BANK_ROOT / 'qwen25_05b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
    '3b': BANK_ROOT / 'qwen25_3b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
}
BIOASQ_FILE = PROJECT_ROOT / 'data/training13b.json'
API_KEY_FILE = PROJECT_ROOT / 'open_ai_api.txt'
CURRENT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/class_judgments/candidate_banks_train_first200_c3_c1_v1'
NEXT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/class_judgments/candidate_banks_train_questions201_400_c3_c1_v1'
MERGED_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/class_judgments/candidate_banks_train_first400_c3_c1_v1'
QUESTION_OFFSET = 200
QUESTION_LIMIT = 200
JUDGE_MODEL = os.environ.get('CANDIDATE_BANK_JUDGE_MODEL', 'gpt-4.1-mini-2025-04-14')
JUDGE_ENDPOINT = os.environ.get('CANDIDATE_BANK_JUDGE_ENDPOINT', 'https://api.openai.com/v1/chat/completions')
MAX_NEW_JUDGE_CALLS = None
MAX_RETRIES = 2
TIMEOUT_SECONDS = 120
PAIR_POLICY = 'all_ordered_class_pairs'

for path in [BIOASQ_FILE, API_KEY_FILE, *BANK_FILES.values(), CURRENT_ROOT / 'candidate_class_judgments.jsonl']:
    assert path.exists(), path
print('Annotating unique question ordinals:', QUESTION_OFFSET + 1, 'to', QUESTION_OFFSET + QUESTION_LIMIT)
print('Expected candidate records:', QUESTION_LIMIT * 20 * len(BANK_FILES))

In [ ]:
# Prepare only the next 200 questions. This is read-only with respect to the banks.
next_records, next_question_ids, next_counts = prepare_records(
    BANK_FILES, BIOASQ_FILE, question_limit=QUESTION_LIMIT, question_offset=QUESTION_OFFSET
)
first_records, first_question_ids, first_counts = prepare_records(
    BANK_FILES, BIOASQ_FILE, question_limit=QUESTION_LIMIT, question_offset=0
)
for model in BANK_FILES:
    assert set(first_question_ids[model]).isdisjoint(next_question_ids[model]), model
    assert len(next_question_ids[model]) == QUESTION_LIMIT
assert len(next_records) == QUESTION_LIMIT * 20 * len(BANK_FILES)
print('Next rows:', len(next_records))
print('Next questions by model:', {k: len(v) for k, v in next_question_ids.items()})
print('Cross-model overlap:', len(set(next_question_ids['05b']) & set(next_question_ids['3b'])))

In [ ]:
# Run or resume the LLM annotation for questions 201–400.
NEXT_ROOT.mkdir(parents=True, exist_ok=True)
judge = CandidateBankClassJudge(
    records=next_records, output_root=NEXT_ROOT, api_key_file=API_KEY_FILE,
    judge_model=JUDGE_MODEL, judge_endpoint=JUDGE_ENDPOINT,
    max_new_judge_calls=MAX_NEW_JUDGE_CALLS, max_retries=MAX_RETRIES,
    timeout_seconds=TIMEOUT_SECONDS,
)
next_judgments, next_summary = judge.run()
print(json.dumps(next_summary, indent=2))

In [ ]:
# Merge first-200 and next-200 judgments in the canonical 400-question record order.
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).open(encoding='utf-8') if line.strip()]

first_judgments = read_jsonl(CURRENT_ROOT / 'candidate_class_judgments.jsonl')
if not (NEXT_ROOT / 'candidate_class_judgments.jsonl').exists():
    raise RuntimeError('Next-block annotation is incomplete; finish the annotation cell first.')
next_judgments = read_jsonl(NEXT_ROOT / 'candidate_class_judgments.jsonl')
merged_records, merged_question_ids, merged_counts = prepare_records(
    BANK_FILES, BIOASQ_FILE, question_limit=400, question_offset=0
)

def key(row):
    return (str(row['question_id']), str(row['source_model']), str(row['response_id']))

judgment_by_key = {}
for row in first_judgments + next_judgments:
    k = key(row)
    if k in judgment_by_key:
        raise ValueError(f'Duplicate judgment key: {k}')
    judgment_by_key[k] = row
merged_judgments = []
missing = []
for record in merged_records:
    value = judgment_by_key.get(key(record))
    if value is None:
        missing.append(key(record))
    else:
        merged_judgments.append(value)
if missing:
    raise ValueError(f'Missing {len(missing)} judgments; first keys: {missing[:5]}')
if len(merged_judgments) != len(merged_records):
    raise ValueError('Merged record/judgment count mismatch')
if MERGED_ROOT.exists():
    raise FileExistsError(f'Refusing to overwrite existing merged output: {MERGED_ROOT}')
MERGED_ROOT.mkdir(parents=True)
(MERGED_ROOT / 'candidate_class_judgments.jsonl').write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in merged_judgments), encoding='utf-8'
)
pair_summary = build_pairs(merged_records, merged_judgments, MERGED_ROOT, policy=PAIR_POLICY)
manifest = {
    'status': 'complete', 'question_offset_first_block': 0,
    'question_offset_next_block': QUESTION_OFFSET, 'question_limit_per_block': QUESTION_LIMIT,
    'merged_question_count': 400, 'candidate_record_count': len(merged_records),
    'source_roots': {'first200': str(CURRENT_ROOT), 'next200': str(NEXT_ROOT)},
    'bank_files': {k: str(v) for k, v in BANK_FILES.items()},
    'judge_model': JUDGE_MODEL, 'judge_endpoint_sha256': digest(JUDGE_ENDPOINT),
    'rubric_sha256': digest(JUDGE_SYSTEM), 'pair_policy': PAIR_POLICY,
    'next_summary': next_summary, 'pair_summary': pair_summary,
}
(MERGED_ROOT / 'summary.json').write_text(json.dumps({**next_summary, **pair_summary, 'merged_question_count': 400}, indent=2) + '\n', encoding='utf-8')
(MERGED_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print(json.dumps(pair_summary, indent=2))
print('Merged output:', MERGED_ROOT)

In [ ]:
# Verify the merged output.
from collections import Counter
merged_j = read_jsonl(MERGED_ROOT / 'candidate_class_judgments.jsonl')
merged_p = read_jsonl(MERGED_ROOT / 'dpo_pairs.jsonl')
print('Merged candidate records:', len(merged_j))
print('Merged unique questions:', len({r['question_id'] for r in merged_j}))
print('Merged class counts:', dict(Counter(r['class'] for r in merged_j)))
print('Merged pairs:', len(merged_p))
print('Merged pair classes:', dict(Counter(f"{r['chosen_class']}>{r['rejected_class']}" for r in merged_p)))